# Understood: fine-tune an extra listener

Runs the whole experiment in order. **Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all**.
Allow Google Drive when asked. Every finished step is saved to Drive and skipped on a rerun, so if Colab
disconnects, just press **Run all** again.

What it does:
1. Scores three untuned models on clips they never trained on.
2. Fine-tunes each on the training clips (all speakers together).
3. Picks the best by words wrong on **unseen prompts** in the dev clips.
4. Personalises the best one per speaker, and keeps that only if it helps.
5. Scores the final choice once on the held-out clips, next to the untuned model.

TORGO audio stays in your private Drive and this Colab session. Keep the folder unshared.

In [ ]:
import torch, subprocess, sys
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU, then Run all again."
print(torch.cuda.get_device_name(0), torch.__version__, sys.version.split()[0])
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, glob, shutil, json, pathlib
BUNDLE = '/content/drive/MyDrive/understood-finetune'   # the folder you uploaded
assert os.path.exists(f'{BUNDLE}/clips.jsonl'), f'Folder not found: {BUNDLE}. Upload understood-finetune to the top of My Drive.'
WORK = '/content/work'
os.makedirs(f'{WORK}/data/audio', exist_ok=True)
for f in glob.glob(f'{BUNDLE}/code/*.py'): shutil.copy(f, WORK)
shutil.copy(f'{BUNDLE}/clips.jsonl', f'{WORK}/data/clips.jsonl')
if len(glob.glob(f'{WORK}/data/audio/*/*.flac')) < 3800:
    for t in sorted(glob.glob(f'{BUNDLE}/torgo-*.tar')): subprocess.run(['tar', '-xf', t, '-C', f'{WORK}/data/audio'], check=True)
print(len(glob.glob(f'{WORK}/data/audio/*/*.flac')), 'clips ready')
RUNS = f'{BUNDLE}/runs'; os.makedirs(RUNS, exist_ok=True)
os.chdir(WORK)

In [ ]:
# Versions pinned to the ones the code was checked with (19 Nov 2026). Torch stays as Colab ships it.
!pip -q install "transformers==5.23.0" "peft==0.21.3" "accelerate>=1.10" soundfile safetensors
try:
    import torchaudio
except ImportError:
    !pip -q install torchaudio
    import torchaudio
import transformers, peft; print(transformers.__version__, peft.__version__, torchaudio.__version__)

In [ ]:
MODELS = {
    # Whisper always works on 30-second windows, so it needs small batches on a T4 (2 x 8 = 16 clips a step).
    'whisper':  dict(id='openai/whisper-large-v3-turbo', method='lora',    extra='--batch-clips 2 --accumulate 8'),
    'parakeet': dict(id='nvidia/parakeet-tdt-0.6b-v3',   method='lora',    extra=''),
    # Granite trains in full 32-bit precision (see asr.py), so its batches are smaller too.
    'granite':  dict(id='ibm-granite/granite-speech-5.0-470m-turboctc', method='encoder', extra='--batch-clips 4 --accumulate 4 --batch-seconds 40'),
}
SEED = 0
failed = []

def step(name, done_file, command, stale=()):
    """Runs one step unless its result is already on Drive. A failed step is noted and the rest go on.
    `stale`: results that depend on this step, deleted when it really runs so they are made again."""
    if os.path.exists(done_file):
        print(f'[skip] {name}: already done'); return True
    for path in stale:
        if os.path.exists(path): os.remove(path)
    print(f'\n===== {name} =====\n{command}')
    get_ipython().system(command)   # shows the output as it comes
    code = get_ipython().user_ns.get('_exit_code', 0)
    if code != 0 or not os.path.exists(done_file):
        failed.append(name); print(f'[FAILED] {name} (exit {code})'); return False
    return True

def scores(path, role='dev'):
    return json.load(open(path))[role]

## 0. Quick check (about 5 minutes)

Trains each model on 16 clips for one epoch, reloads it and transcribes 8 clips. This proves every code path works on this GPU before the long steps. A model that fails here is left out, and the others go on.


In [ ]:
QUICK = '/content/quick'
for name, m in list(MODELS.items()):
    q = f'{QUICK}/{name}'
    ok = step(f'quick train {name}', f'{q}/done.json', f"python finetune.py --model {m['id']} --method {m['method']} {m['extra']} --limit 16 --epochs 1 --out {q}")
    ok = ok and step(f'quick reload {name}', f'{q}/scores-val.json', f"python evaluate.py --run {q} --roles val --limit 8 --out {q}")
    ok = ok and step(f'quick personalise {name}', f'{q}-s2/F01/done.json', f"python finetune.py --parent {q} --speaker F01 {m['extra']} --limit 8 --epochs 1 --out {q}-s2/F01")
    ok = ok and step(f'quick personalised reload {name}', f'{q}-s2/scores-val.json', f"python evaluate.py --per-speaker-runs {q}-s2 --fallback-run {q} --roles val --limit 8 --out {q}-s2")
    if not ok:
        print(f'>>> {name} is left out of this run'); MODELS.pop(name)
assert MODELS, 'Every model failed the quick check. Send Claude the errors above.'
print('Models going ahead:', list(MODELS))


## 1. Untuned models

In [ ]:
for name, m in MODELS.items():
    step(f'baseline {name}', f'{RUNS}/base-{name}/scores-dev-val.json',
         f"python evaluate.py --model {m['id']} --roles val,dev --out {RUNS}/base-{name}")

## 2. Fine-tune each model on all speakers

In [ ]:
for name, m in MODELS.items():
    ok = step(f'train {name}', f'{RUNS}/{name}-s1/done.json',
              f"python finetune.py --model {m['id']} --method {m['method']} --seed {SEED} {m['extra']} --out {RUNS}/{name}-s1",
              stale=[f'{RUNS}/{name}-s1/scores-dev.json'])
    if ok:
        step(f'score {name}', f'{RUNS}/{name}-s1/scores-dev.json', f"python evaluate.py --run {RUNS}/{name}-s1 --roles dev --out {RUNS}/{name}-s1")

## 3. Compare on the dev clips and pick one

In [ ]:
rows = []
for name in MODELS:
    for tag, path in (('untuned', f'{RUNS}/base-{name}/scores-dev-val.json'), ('tuned', f'{RUNS}/{name}-s1/scores-dev.json')):
        if os.path.exists(path) and (tag == 'untuned' or os.path.exists(f'{RUNS}/{name}-s1/done.json')):
            s = scores(path); rows.append((name, tag, s['unseen']['wer'], s['all']['wer'], s.get('dysarthric-unseen', {}).get('wer'), s['word']['wer'], s['sentence']['wer']))
print(f"{'model':10}{'':9}{'unseen':>8}{'all':>8}{'dys-unseen':>12}{'words':>8}{'sentences':>11}")
for r in rows: print(f"{r[0]:10}{r[1]:9}" + ''.join(f"{(x or 0)*100:>{w}.1f}" for x, w in zip(r[2:], (8, 8, 12, 8, 11))))
tuned = [r for r in rows if r[1] == 'tuned']
assert tuned, 'No model finished training. See the errors above.'
BEST = min(tuned, key=lambda r: r[2])[0]   # fewest words wrong on unseen prompts
json.dump({'best': BEST, 'rows': rows}, open(f'{RUNS}/choice.json', 'w'), indent=2)
print('\nChosen:', BEST)

## 4. Personalise the chosen model per speaker

In [ ]:
speakers = sorted({json.loads(l)['speaker'] for l in open('data/clips.jsonl')})
s2_scores = f'{RUNS}/{BEST}-s2/scores-dev.json'
for sp in speakers:
    step(f'personalise {sp}', f'{RUNS}/{BEST}-s2/{sp}/done.json',
         f"python finetune.py --parent {RUNS}/{BEST}-s1 --speaker {sp} --seed {SEED} {MODELS[BEST]['extra']} --out {RUNS}/{BEST}-s2/{sp}", stale=[s2_scores])
# A speaker whose run did not finish is heard by the all-speakers model instead.
for sp in speakers:
    if not os.path.exists(f'{RUNS}/{BEST}-s2/{sp}/done.json') and os.path.exists(f'{RUNS}/{BEST}-s2/{sp}/config.json'): os.remove(f'{RUNS}/{BEST}-s2/{sp}/config.json')
step('score personalised', s2_scores,
     f"python evaluate.py --per-speaker-runs {RUNS}/{BEST}-s2 --fallback-run {RUNS}/{BEST}-s1 --roles dev --out {RUNS}/{BEST}-s2")
s1 = scores(f'{RUNS}/{BEST}-s1/scores-dev.json')
s2 = scores(s2_scores) if os.path.exists(s2_scores) else None
print('dev, unseen prompts: all speakers together', s1['unseen']['wer'], ' personalised', s2 and s2['unseen']['wer'])
print('dev, all prompts:    all speakers together', s1['all']['wer'], ' personalised', s2 and s2['all']['wer'])
# Personalising is kept only if it does not hurt unseen prompts and helps overall.
USE_S2 = bool(s2) and s2['unseen']['wer'] <= s1['unseen']['wer'] + 0.005 and s2['all']['wer'] < s1['all']['wer']
print('Keep personalised models:', USE_S2)

## 5. Final score on the held-out clips (once)

In [ ]:
m = MODELS[BEST]
TAG = f"{BEST}-{'s2' if USE_S2 else 's1'}"   # the result folders name the setup, so a different choice is never mixed up with this one
FINAL_UNTUNED, FINAL_TUNED = f'{RUNS}/final-untuned-{BEST}', f'{RUNS}/final-tuned-{TAG}'
step('held-out, untuned', f'{FINAL_UNTUNED}/scores-holdout.json', f"python evaluate.py --model {m['id']} --roles holdout --out {FINAL_UNTUNED}")
final_cmd = (f"python evaluate.py --per-speaker-runs {RUNS}/{BEST}-s2 --fallback-run {RUNS}/{BEST}-s1" if USE_S2 else f"python evaluate.py --run {RUNS}/{BEST}-s1")
step('held-out, tuned', f'{FINAL_TUNED}/scores-holdout.json', f"{final_cmd} --roles holdout --out {FINAL_TUNED}")

In [ ]:
lines = ['# Fine-tuning result', '', f'Chosen model: **{MODELS[BEST]["id"]}** ({"personalised per speaker" if USE_S2 else "one model for all speakers"}).', '']
paths = [f'{FINAL_UNTUNED}/scores-holdout.json', f'{FINAL_TUNED}/scores-holdout.json']
if all(os.path.exists(p) for p in paths):
    a, b = (scores(p, 'holdout') for p in paths)
    lines += ['Share of words wrong on the held-out clips (95% interval). Lower is better.', '', '| Clips | Untuned | Tuned |', '|---|---|---|']
    label = {'unseen': 'Prompts never trained on', 'unseen-strict': 'Never trained on, and not a validation prompt', 'dysarthric-unseen': 'Never trained on, speakers with dysarthria',
             'all': 'All held-out clips', 'seen': 'Prompts also in training', 'dysarthric': 'Speakers with dysarthria', 'control': 'Speakers without',
             'unseen-word': 'Single words, never trained on', 'unseen-sentence': 'Sentences, never trained on'}
    fmt = lambda s: f"{s['wer']*100:.1f}% ({s['low']*100:.1f}-{s['high']*100:.1f})"
    for k, name in label.items():
        if k in a and k in b: lines.append(f'| {name} ({a[k]["clips"]}) | {fmt(a[k])} | {fmt(b[k])} |')
    lines += ['', 'Per speaker, tuned: ' + ', '.join(f"{k[8:]} {v['wer']*100:.0f}%" for k, v in b.items() if k.startswith('speaker:')),
              '', f'Speed: {json.load(open(paths[1]))["seconds_per_clip"]} s per clip on this GPU.']
else:
    lines += ['The held-out scoring did not finish. See the failed steps.']
lines += ['', 'Dev comparison: see choice.json.', '', 'Failed steps: ' + (', '.join(failed) or 'none')]
open(f'{RUNS}/RESULTS.md', 'w').write('\n'.join(lines)); print('\n'.join(lines))

Done. Everything is in `My Drive/understood-finetune/runs`. Tell Claude it finished.